
## Overview

This notebook will show you how to create and query a table or DataFrame that you uploaded to DBFS. [DBFS](https://docs.databricks.com/user-guide/dbfs-databricks-file-system.html) is a Databricks File System that allows you to store data for querying inside of Databricks. This notebook assumes that you have a file already inside of DBFS that you would like to read from.

This notebook is written in **Python** so the default cell type is Python. However, you can use different languages by using the `%LANGUAGE` syntax. Python, Scala, SQL, and R are all supported.

In [0]:
# File location and type
file_location = "/FileStore/tables/tips.csv"
file_type = "csv"

# CSV options

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.csv(file_location, header = True, inferSchema = True)

In [0]:
df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|
|     18.43| 3.0|  Male|    No|Sun|Dinner|   4|
|     14.83|3.02|Female|    No|Sun|Dinner|   2|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     10.33|1.67|Female|    No|Sun|Dinner|   3|
|     16.29|3.71|  Male|    No|Sun|Dinne

In [0]:
df.printSchema()

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)



In [0]:
df.columns

Out[5]: ['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']

In [0]:
#  handling categorical features
from pyspark.ml.feature import StringIndexer

In [0]:
index = StringIndexer(inputCol="sex", outputCol="sex_indexed")
df_r = index.fit(df).transform(df)
df_r.show()

+----------+----+------+------+---+------+----+-----------+
|total_bill| tip|   sex|smoker|day|  time|size|sex_indexed|
+----------+----+------+------+---+------+----+-----------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|        1.0|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|        0.0|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|        0.0|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|        0.0|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|        1.0|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|        0.0|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|        0.0|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|        0.0|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|        0.0|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|        0.0|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|        0.0|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|        1.0|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|        0.0|
|     18.43| 3.0|  Male|    No|Sun|Dinne

In [0]:
index = StringIndexer(inputCols=["smoker",'day','time'], outputCols=["smoker_indexed","day_indexed","time_indexed"])
df_r = index.fit(df_r).transform(df_r)
df_r.show()

+----------+----+------+------+---+------+----+-----------+--------------+-----------+------------+
|total_bill| tip|   sex|smoker|day|  time|size|sex_indexed|smoker_indexed|day_indexed|time_indexed|
+----------+----+------+------+---+------+----+-----------+--------------+-----------+------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|        1.0|           0.0|        1.0|         0.0|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|        0.0|           0.0|        1.0|         0.0|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|        0.0|           0.0|        1.0|         0.0|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|        0.0|           0.0|        1.0|         0.0|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|        1.0|           0.0|        1.0|         0.0|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|        0.0|           0.0|        1.0|         0.0|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|        0.0|           0.0|        1.0|         0.0|


In [0]:
df_r.columns

Out[12]: ['total_bill',
 'tip',
 'sex',
 'smoker',
 'day',
 'time',
 'size',
 'sex_indexed',
 'smoker_indexed',
 'day_indexed',
 'time_indexed']

In [0]:
from pyspark.ml.feature import VectorAssembler

featureAssambler = VectorAssembler(inputCols= ['tip','sex_indexed','smoker_indexed','day_indexed','time_indexed',"size"], outputCol = "Independent Features")

output = featureAssambler.transform(df_r)

In [0]:
output.show()

+----------+----+------+------+---+------+----+-----------+--------------+-----------+------------+--------------------+
|total_bill| tip|   sex|smoker|day|  time|size|sex_indexed|smoker_indexed|day_indexed|time_indexed|Independent Features|
+----------+----+------+------+---+------+----+-----------+--------------+-----------+------------+--------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|        1.0|           0.0|        1.0|         0.0|[1.01,1.0,0.0,1.0...|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|        0.0|           0.0|        1.0|         0.0|[1.66,0.0,0.0,1.0...|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|        0.0|           0.0|        1.0|         0.0|[3.5,0.0,0.0,1.0,...|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|        0.0|           0.0|        1.0|         0.0|[3.31,0.0,0.0,1.0...|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|        1.0|           0.0|        1.0|         0.0|[3.61,1.0,0.0,1.0...|
|     25.29|4.71|  Male|    No|S

In [0]:
output.select("total_bill",'Independent Features').show()

+----------+--------------------+
|total_bill|Independent Features|
+----------+--------------------+
|     16.99|[1.01,1.0,0.0,1.0...|
|     10.34|[1.66,0.0,0.0,1.0...|
|     21.01|[3.5,0.0,0.0,1.0,...|
|     23.68|[3.31,0.0,0.0,1.0...|
|     24.59|[3.61,1.0,0.0,1.0...|
|     25.29|[4.71,0.0,0.0,1.0...|
|      8.77|[2.0,0.0,0.0,1.0,...|
|     26.88|[3.12,0.0,0.0,1.0...|
|     15.04|[1.96,0.0,0.0,1.0...|
|     14.78|[3.23,0.0,0.0,1.0...|
|     10.27|[1.71,0.0,0.0,1.0...|
|     35.26|[5.0,1.0,0.0,1.0,...|
|     15.42|[1.57,0.0,0.0,1.0...|
|     18.43|[3.0,0.0,0.0,1.0,...|
|     14.83|[3.02,1.0,0.0,1.0...|
|     21.58|[3.92,0.0,0.0,1.0...|
|     10.33|[1.67,1.0,0.0,1.0...|
|     16.29|[3.71,0.0,0.0,1.0...|
|     16.97|[3.5,1.0,0.0,1.0,...|
|     20.65|(6,[0,5],[3.35,3.0])|
+----------+--------------------+
only showing top 20 rows



In [0]:
finalizedData = output.select("total_bill",'Independent Features')

In [0]:
finalizedData.show()

+----------+--------------------+
|total_bill|Independent Features|
+----------+--------------------+
|     16.99|[1.01,1.0,0.0,1.0...|
|     10.34|[1.66,0.0,0.0,1.0...|
|     21.01|[3.5,0.0,0.0,1.0,...|
|     23.68|[3.31,0.0,0.0,1.0...|
|     24.59|[3.61,1.0,0.0,1.0...|
|     25.29|[4.71,0.0,0.0,1.0...|
|      8.77|[2.0,0.0,0.0,1.0,...|
|     26.88|[3.12,0.0,0.0,1.0...|
|     15.04|[1.96,0.0,0.0,1.0...|
|     14.78|[3.23,0.0,0.0,1.0...|
|     10.27|[1.71,0.0,0.0,1.0...|
|     35.26|[5.0,1.0,0.0,1.0,...|
|     15.42|[1.57,0.0,0.0,1.0...|
|     18.43|[3.0,0.0,0.0,1.0,...|
|     14.83|[3.02,1.0,0.0,1.0...|
|     21.58|[3.92,0.0,0.0,1.0...|
|     10.33|[1.67,1.0,0.0,1.0...|
|     16.29|[3.71,0.0,0.0,1.0...|
|     16.97|[3.5,1.0,0.0,1.0,...|
|     20.65|(6,[0,5],[3.35,3.0])|
+----------+--------------------+
only showing top 20 rows



In [0]:
from pyspark.ml.regression import LinearRegression
# train test split
train_data,test_data = finalizedData.randomSplit([0.75,0.25],42)
regressor = LinearRegression(featuresCol="Independent Features", labelCol= "total_bill")
regressor = regressor.fit(train_data)

In [0]:
regressor.coefficients

Out[35]: DenseVector([3.1745, -0.6843, 2.0909, -0.4913, -0.4759, 3.1265])

In [0]:
regressor.intercept

Out[36]: 2.502905990827671

In [0]:
predResult = regressor.evaluate(test_data)
predResult.predictions.show()

+----------+--------------------+------------------+
|total_bill|Independent Features|        prediction|
+----------+--------------------+------------------+
|      7.25|[1.0,1.0,0.0,0.0,...|  8.11969969390293|
|      7.74|[1.44,0.0,1.0,0.0...|15.418221143057956|
|      8.51|[1.25,1.0,0.0,2.0...|10.581369112360315|
|      8.52|[1.48,0.0,0.0,2.0...|11.995798005920484|
|       9.6|[4.0,1.0,1.0,1.0,...|22.369439650408808|
|     10.09|[2.0,1.0,1.0,3.0,...|  14.5618589038155|
|     10.33|[2.0,1.0,0.0,2.0,...|12.962271736195069|
|     10.65|[1.5,1.0,0.0,2.0,...|11.375003320305233|
|     10.77|(6,[0,5],[1.47,2.0])| 13.42255972063665|
|     11.17|[1.5,1.0,0.0,2.0,...|11.375003320305233|
|     11.35|[2.5,1.0,1.0,3.0,...| 16.62501368323077|
|     11.38|[2.0,1.0,0.0,2.0,...|12.962271736195069|
|     12.16|[2.2,0.0,1.0,3.0,...|15.881051692422279|
|     12.46|[1.5,0.0,0.0,3.0,...|12.043864746327243|
|     12.48|[2.52,1.0,0.0,2.0...|14.613030888720498|
|     12.54|[2.5,0.0,0.0,1.0,...| 16.201022297

In [0]:
predResult.meanAbsoluteError, predResult.meanSquaredError

Out[38]: (3.4590471457320313, 21.50203696636162)